# Лабораторная работа №1. Предварительный анализ данных
Курбатов М. П., группа 4417.

Цель: проверить качество исходного CSV, выполнить очистку, группировки и сводные таблицы.

Набор movies.csv содержит сведения о фильмах: title — название, release_year — год выхода, score — рейтинг, number_of_votes — число голосов, duration — длительность, main_genre — основной жанр, main_production — страна производства. Вариант №13.

## 1. Чтение файла
В Colab загрузите CSV в папку /content и при необходимости измените путь.

In [1]:
import pandas as pd
file_path = '/content/movies.csv'
df = pd.read_csv(file_path, sep=';')
initial_rows = len(df)
print('Размер исходной таблицы:', df.shape)

Размер исходной таблицы: (389, 7)


## 2. Обзор данных
Вывожу первые 20 строк, информацию о типах и описательную статистику числовых столбцов.

In [2]:
display(df.head(20))
df.info()
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 389 entries, 0 to 388
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            384 non-null    object 
 1   release_year     387 non-null    float64
 2   score            386 non-null    float64
 3   number_of_votes  388 non-null    float64
 4   duration         389 non-null    int64  
 5   main_genre       389 non-null    object 
 6   main_production  388 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 21.4+ KB


,title,release_year,score,number_of_votes,duration,main_genre,main_production
0,David Attenborough: A Life on Our Planet,2020.0,9.0,31180.0,83,documentary,GB
1,Inception,2010.0,8.8,2268288.0,148,scifi,GB
2,Forrest Gump,1994.0,8.8,1994599.0,142,drama,US
3,Anbe Sivam,2003.0,8.7,20595.0,160,comedy,IN
4,Bo Burnham: Inside,2021.0,8.7,44074.0,87,comedy,US
5,Saving Private Ryan,1998.0,8.6,1346020.0,169,drama,US
6,Django Unchained,2012.0,8.4,1472668.0,165,western,US
7,Dangal,2016.0,8.4,180247.0,161,action,IN
8,Bo Burnham: Make Happy,2016.0,8.4,14356.0,60,comedy,US
9,Louis C.K.: Hilarious,2010.0,8.4,11973.0,84,comedy,US


,release_year,score,number_of_votes,duration
count,387.000000,386.000000,3.880000e+02,389.000000
mean,2010.976744,7.512176,1.391523e+05,123.352185
std,10.805726,0.443870,2.365279e+05,28.304800
min,1954.000000,6.900000,1.013900e+04,28.000000
25%,2007.500000,7.100000,2.055375e+04,104.000000
50%,2014.000000,7.400000,4.606950e+04,122.000000
75%,2018.000000,7.800000,1.551585e+05,139.000000
max,2022.000000,9.000000,2.268288e+06,229.000000


### Названия столбцов
columns — атрибут DataFrame, а не метод. Он возвращает Index названий столбцов. Названия должны быть понятными и не содержать случайных пробелов.

In [3]:
print(df.columns.tolist())
df.columns = df.columns.str.strip()
print('Повторяющихся названий:', df.columns.duplicated().sum())

['title', 'release_year', 'score', 'number_of_votes', 'duration', 'main_genre', 'main_production']
Повторяющихся названий: 0


## 3. Пропуски
Строки с пропусками удаляются: их немного, а названия фильмов и страны нельзя надёжно восстановить средним значением. Считаю число таких строк отдельно от общего числа пустых ячеек.

In [4]:
display(df.isna().sum().to_frame('Пропусков'))
missing_rows = df.isna().any(axis=1).sum()
print('Строк с пропусками:', missing_rows)
df = df.dropna().copy()
print('После удаления пропусков:', df.shape)

Строк с пропусками: 6
После удаления пропусков: (383, 7)


,Пропусков
title,5
release_year,2
score,3
number_of_votes,1
duration,0
main_genre,0
main_production,1


## 4. Явные и неявные дубликаты
Явные дубликаты — полностью одинаковые строки. При проверке категорий ищу различия регистра и опечатки. Повторяющиеся страны и жанры сами по себе дубликатами фильмов не являются.

In [5]:
print('Явных дубликатов:', df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Жанры до нормализации:', sorted(df['main_genre'].unique()))
print('Страны до нормализации:', sorted(df['main_production'].unique()))
df['main_genre'] = df['main_genre'].str.strip().str.lower().replace('dramaa', 'drama')
df['main_production'] = df['main_production'].str.strip().str.upper()
print('Жанры после нормализации:', sorted(df['main_genre'].unique()))
print('Повторных строк после нормализации:', df.duplicated().sum())
df = df.drop_duplicates().copy()

Явных дубликатов: 2
Жанры до нормализации: ['COMEDY', 'action', 'animation', 'comedy', 'crime', 'documentary', 'drama', 'dramaa', 'fantasy', 'horror', 'musical', 'romance', 'scifi', 'sports', 'thriller', 'war', 'western']
Страны до нормализации: ['AR', 'AU', 'BE', 'BR', 'CA', 'CD', 'CN', 'DE', 'DK', 'ES', 'FR', 'GB', 'HK', 'HU', 'ID', 'IE', 'IN', 'IT', 'JP', 'KH', 'KR', 'LT', 'MW', 'MX', 'NL', 'NO', 'NZ', 'PL', 'PS', 'TH', 'TR', 'UA', 'US', 'XX', 'ZA']
Жанры после нормализации: ['action', 'animation', 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'horror', 'musical', 'romance', 'scifi', 'sports', 'thriller', 'war', 'western']
Повторных строк после нормализации: 0


## 5. Типы данных
Год и число голосов должны содержать целые значения. Перед astype проверяю, что дробной части нет. Рейтинг оставляю вещественным, поскольку он может содержать десятичную часть.

In [6]:
for column in ['release_year', 'number_of_votes']:
    assert (df[column] % 1 == 0).all(), f'В {column} обнаружена дробная часть'
    df[column] = df[column].astype(int)
df.info()
display(df.describe())
print('Итоговый размер:', df.shape)
print('Пропусков:', df.isna().sum().sum())
print('Явных дубликатов:', df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 381 entries, 0 to 386
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            381 non-null    object 
 1   release_year     381 non-null    int64  
 2   score            381 non-null    float64
 3   number_of_votes  381 non-null    int64  
 4   duration         381 non-null    int64  
 5   main_genre       381 non-null    object 
 6   main_production  381 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 23.8+ KB
Итоговый размер: (381, 7)
Пропусков: 0
Явных дубликатов: 0


,release_year,score,number_of_votes,duration
count,381.000000,381.000000,3.810000e+02,381.000000
mean,2010.997375,7.509186,1.362626e+05,123.530184
std,10.862659,0.445084,2.350996e+05,28.528329
min,1954.000000,6.900000,1.013900e+04,28.000000
25%,2008.000000,7.100000,2.043000e+04,104.000000
50%,2014.000000,7.400000,4.491700e+04,122.000000
75%,2018.000000,7.800000,1.534630e+05,139.000000
max,2022.000000,9.000000,2.268288e+06,229.000000


## 6. Группировки
### Задание 1
Посчитать количество фильмов для сочетаний жанра и страны производства. 

In [7]:
group1 = df.groupby(['main_genre', 'main_production']).size()
with pd.option_context('display.max_rows', None):
    display(group1.to_frame('Количество'))
print('Число групп:', len(group1))

Число групп: 93


Количество
main_genre  main_production            
action      DE                        1
            IN                        2
            US                        2
animation   JP                        1
            US                        2
comedy      AR                        1
            AU                        1
            CA                        1
            ES                        1
            FR                        1
            GB                        5
            IN                       20
            JP                        1
            NO                        1
            NZ                        1
            TR                        2
            US                       21
            XX                        1
crime       DK                        1
            GB                        2
            IN                        9
            NO                        1
            US                        8
documentary BE                        1
            BR                        1
            GB                        1
            UA                        2
            US                       14
            ZA                        1
drama       AU                        1
            CA                        2
            CN                        2
            DE                        4
            ES                        1
            FR                        2
            GB                       14
            HK                        2
            HU                        1
            IE                        1
            IN                       48
            IT                        3
            JP                        1
            KH                        1
            KR                        2
            LT                        1
            MW                        1
            MX                        1
            NL                        2
            NZ                        1
            TR                        4
            US                       53
fantasy     CN                        1
            ES                        1
            FR                        1
            HU                        1
            IN                        2
            IT                        1
            JP                        3
            PL                        1
            US                        8
horror      BE                        1
            DE                        1
            ES                        1
            GB                        1
            IN                        1
            JP                        1
            US                        2
musical     US                        3
romance     GB                        1
            IN                       12
            JP                        1
            US                        6
scifi       CA                        1
            GB                        2
            JP                        1
            TR                        1
            US                        2
sports      DE                        1
thriller    AU                        1
            DE                        2
            ES                        2
            FR                        3
            GB                        3
            ID                        1
            IE                        1
            IN                       21
            KR                        2
            PS                        1
            TH                        1
            US                       21
war         CD                        1
            US                        2
western     US                        7

Получено 93 сочетания жанра и страны. Один жанр может быть представлен несколькими странами.

### Задание 2
Преобразовать результат группировки в DataFrame, назвать столбец количества count и отсортировать по убыванию.

In [8]:
group2 = group1.reset_index(name='count').sort_values('count', ascending=False)
with pd.option_context('display.max_rows', None):
    display(group2)

,main_genre,main_production,count
50,drama,US,53
39,drama,IN,48
89,thriller,US,21
85,thriller,IN,21
16,comedy,US,21
11,comedy,IN,20
27,documentary,US,14
35,drama,GB,14
69,romance,IN,12
20,crime,IN,9


Первые строки отсортированной таблицы — drama/US (53 фильма) и drama/IN (48 фильмов). Столбец count показывает число наблюдений, а не число уникальных стран.

### Задание 3
Построить сводную таблицу среднего рейтинга по странам. Сначала сортирую по исходному среднему, затем округляю до одного десятичного знака.

In [9]:
pivot_mean = df.pivot_table(index='main_production', values='score', aggfunc='mean')
pivot_mean = pivot_mean.sort_values('score', ascending=False).round(1)
display(pivot_mean)

,score
main_production,
CD,8.2
UA,8.1
ZA,8.1
TR,7.8
NZ,7.8
MX,7.7
TH,7.6
MW,7.6
IN,7.6


Наибольшее среднее значение: CD — 8,2; далее UA и ZA — 8,1. Результат зависит от числа фильмов каждой страны.

### Задание 4
Построить сводную таблицу медианного рейтинга: страны — строки, жанры — столбцы. Отсортировать страны по возрастанию.

In [10]:
pivot_median = df.pivot_table(
    index='main_production',
    columns='main_genre',
    values='score',
    aggfunc='median'
).sort_index()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pivot_median)

main_genre,action,animation,comedy,crime,documentary,drama,fantasy,horror,musical,romance,scifi,sports,thriller,war,western
main_production,,,,,,,,,,,,,,,
AR,NaN,NaN,7.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AU,NaN,NaN,8.10,NaN,NaN,7.30,NaN,NaN,NaN,NaN,NaN,NaN,7.30,NaN,NaN
BE,NaN,NaN,NaN,NaN,7.9,NaN,NaN,7.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BR,NaN,NaN,NaN,NaN,7.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CA,NaN,NaN,7.10,NaN,NaN,7.15,NaN,NaN,NaN,NaN,8.00,NaN,NaN,NaN,NaN
CD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.2,NaN
CN,NaN,NaN,NaN,NaN,NaN,7.25,7.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DE,7.10,NaN,NaN,NaN,NaN,7.15,NaN,7.50,NaN,NaN,NaN,7.4,7.75,NaN,NaN
DK,NaN,NaN,NaN,6.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Медианные рейтинги драм Индии и США равны 7,6 и 7,4. Пустые ячейки показывают отсутствующие сочетания страны и жанра.

NaN в сводной таблице означает отсутствие наблюдений для конкретной пары страны и жанра, а не новый пропуск в очищенном наборе. Средние и медианы небольших групп следует интерпретировать осторожно: сравнение стран не доказывает причинного влияния страны на рейтинг.

## Общий вывод
Выполнен предварительный анализ movies.csv. Исходно в таблице 389 строк и 7 столбцов. Удалено 6 строк с пропусками и 2 явных дубликата; осталось 381 наблюдение. Жанры приведены к нижнему регистру, исправлена опечатка dramaa, коды стран нормализованы. Год выхода и число голосов преобразованы в int после проверки отсутствия дробных значений.

Группировка по жанрам и странам дала 93 сочетания. Наиболее многочисленны драмы США — 53 фильма и Индии — 48. Сводная таблица средних рейтингов по странам показала максимум 8,2 для CD; для UA и ZA среднее равно 8,1. Средние небольших групп не следует считать оценкой всего кинематографа страны.

Медианные рейтинги драм Индии и США равны 7,6 и 7,4, комедий — 7,6 и 7,3 соответственно. NaN в сводной таблице означает отсутствие сочетания жанра и страны в данных, а не пропуск, оставленный при очистке. Набор приведён к согласованному виду и подготовлен для дальнейшего анализа.